In [ ]:
import sys
import os

current_dir = os.getcwd()
src_dir = os.path.dirname(current_dir)
sys.path.append(src_dir)

import polars as pl
from pipeline.config import PipelineConfig, DataConfig, ProcessingConfig, StorageConfig, RayConfig, S3Location
from pipeline.pipeline import Pipeline
from data_preprocessing.data_normalization import BMLLNormalizer
from data_preprocessing.data_access import DataAccessFactory

In [ ]:
# 1. Create pipeline with retry configuration
config = PipelineConfig(
    region='us-east-1',
    data=DataConfig(
        raw_data_path='s3://bmlldata',
        start_date='2024-01-02',
        end_date='2024-01-02',
        exchanges=['ARCX'],
        data_types=['trades']
    ),
    processing=ProcessingConfig(
        normalization=BMLLNormalizer()
    ),
    storage=StorageConfig(
        raw_data=S3Location(path='s3://bmlldata'),
        normalized=S3Location(path='s3://orderflowanalysis/intermediate/normalized'),
        features=S3Location(path='s3://orderflowanalysis/intermediate/features'),
        models=S3Location(path='s3://orderflowanalysis/output/models'),
        predictions=S3Location(path='s3://orderflowanalysis/output/predictions'),
        backtest=S3Location(path='s3://orderflowanalysis/output/backtest')
    ),
    ray=RayConfig(runtime_env={"working_dir": src_dir}, max_retries=5, cpu_buffer=4, file_sort_order="desc", max_pending_tasks=2500)
)

pipeline = Pipeline(config)

In [ ]:
# 2. Run pipeline
results = None
results = pipeline.run(slice(100))
print(f"Processed {len(results)} files")

In [ ]:
# 3. Verify retry functionality
import pandas as pd
res_pd = pd.DataFrame(results)
successful = res_pd.query("message == 'success'").shape[0]
failed = res_pd.query("message != 'success'").shape[0]
print(f"Successful: {successful}, Failed: {failed}")

In [ ]:
# 4. Inspect paths
result = results[0]
print(f"Input:  {result['input_path']}")
print(f"Output: {result['output_path']}")
print(f"Rows:   {result['row_count']:,}")

In [ ]:
# 5. Read output
data_access = DataAccessFactory.create('s3', region='us-east-1')
output_data = data_access.read(result['output_path']).limit(100).collect()
print(f"✓ Read {len(output_data)} records")
output_data.head()

In [ ]:
# 6. Validate schema
normalizer = BMLLNormalizer()
expected_schema = normalizer.get_schema('trades')
missing = set(expected_schema.keys()) - set(output_data.columns)
assert not missing, f"Missing columns: {missing}"
print(f"✓ Schema validated ({len(output_data.columns)} columns)")
print("\n✓ All tests passed!")